# DocStruct — section-boundary run (Colab T4)

Upload this notebook, set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Nothing else to do. Every cell is idempotent.

## It is meant to be run more than once

Free Colab reclaims sessions without warning. Everything expensive lives on Drive: the
corpora, the model weights, the detector cache, the benchmark checkpoints, the PDF text
spines and the section checkpoints. **If the session dies, reopen and Run all again** —
every stage skips what is already done. Re-running a finished run is cheap.

## What it produces

1. **Section-boundary agreement** — Pk / WindowDiff / straddle rate for every chunker
   against 126 papers' publisher-authored JATS gold, including hybrid `docstruct`,
   which is too slow to run on a laptop (one figure-dense paper measured 475 s
   geometry-only on CPU). This is the run that matters.
2. **The section-boundary ceiling**, so the scores are read against what is reachable
   at all rather than against 100%.

**FinanceBench retrieval is off by default.** Measured on 2026-08-13, its evidence is
unreachable at top-5 under three different embedders — every tool scores 0.0, which
measures the retriever rather than the chunkers. Section 7 carries the numbers and the
switch to reproduce it.

## 1. GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "NO GPU - Runtime > Change runtime type > T4 GPU, then Run all again."
print('GPU ok:', torch.cuda.get_device_name(0))

# Record the Pillow this kernel has ALREADY loaded, before anything pip-installs over
# it. Colab imports PIL at startup, and PIL._imaging is a compiled C extension: once it
# is in the process it cannot be unloaded, not by deleting sys.modules entries and not
# by reimporting. If pip later upgrades the Python files underneath it, every PIL import
# dies with "The _imaging extension was built for another version of Pillow". Pinning
# back to this exact version after the install keeps the files and the loaded extension
# in agreement, which is what avoids a runtime restart mid-"Run all".
import PIL
PRELOADED_PILLOW = PIL.__version__
print('pillow already loaded in this kernel:', PRELOADED_PILLOW)

## 2. Drive — every expensive artefact lives here

`BENCH` is the single directory this notebook owns. Deleting it resets everything;
leaving it alone is what makes a re-run resume.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BENCH    = '/content/drive/MyDrive/docstruct_bench'
CORPORA  = BENCH + '/corpora'        # fetched PDFs, so a dead session never refetches
CACHE    = BENCH + '/.bench_cache'   # benchmark checkpoints + detector proposals
DOTCACHE = BENCH + '/.cache'         # PDF text spines + section-scorer checkpoints
WEIGHTS  = BENCH + '/weights'
REPORTS  = BENCH + '/reports'
for d in (BENCH, CORPORA, CACHE, DOTCACHE, WEIGHTS, REPORTS):
    os.makedirs(d, exist_ok=True)
print('\n'.join((BENCH, CORPORA, CACHE, DOTCACHE, WEIGHTS, REPORTS)))

## 3. Repo and dependencies

**Ignore pip's dependency-conflict warnings** about `jedi`, `requests`/`google-colab`,
`opentelemetry`/`google-adk` and the `huggingface-hub` `inference` extra. Pip reports on
the whole environment, including Colab preinstalls this benchmark never imports. Only a
traceback matters.

The one that *is* fatal is Pillow. Colab imports PIL at kernel startup, so the compiled
`PIL._imaging` extension is loaded into the process before anything here runs, and a C
extension cannot be unloaded. If the install below upgrades Pillow's Python files
underneath it, every later `from PIL import Image` raises *"The _imaging extension was
built for another version of Pillow"* and takes ultralytics down with it. Section 1
recorded the loaded version; the install pins it straight back.

In [ ]:
%cd /content
if not os.path.exists('/content/DocStruct/.git'):
    !git clone -q -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git pull -q --ff-only 2>/dev/null || echo '(could not fast-forward; keeping what is here)'
!git log --oneline -1

In [ ]:
!pip install -q -e ".[all,benchmark-heavy]" unstructured-inference pyarrow    llama-index-core llama-index-embeddings-huggingface

# Put Pillow back to the version this kernel loaded at startup (see section 1). The
# alternative -- upgrading and restarting the runtime -- cannot be done inside Run all.
!pip install -q "pillow=={PRELOADED_PILLOW}"

import PIL
from PIL import Image
Image.new('RGB', (4, 4))
print('pillow', PIL.__version__, 'ok (core and files agree)')

### Point the caches at Drive

`.cache/` holds the PDF text spines and the section-scorer checkpoints. Symlinking the
whole directory means a killed run resumes without any flag being passed.

In [ ]:
import os
if not os.path.islink('/content/DocStruct/.cache'):
    !rm -rf /content/DocStruct/.cache
    os.symlink(DOTCACHE, '/content/DocStruct/.cache')
print('.cache ->', os.path.realpath('/content/DocStruct/.cache'))

In [ ]:
# Which adapters actually imported. get_adapters() swallows import errors and drops
# anything whose available() is False, so a missing dependency would silently shrink the
# leaderboard instead of failing. Read the MISSING line before the long runs.
from docstruct.eval.adapters import get_adapters
WANT = ['docstruct', 'docstruct_geo', 'langchain', 'pymupdf4llm',
        'unstructured', 'llamaindex', 'llamaindex_semantic']
got = get_adapters(names=WANT, weights=None)
print('available:', sorted(got))
print('MISSING  :', sorted(set(WANT) - set(got)))
TOOLS = ','.join(n for n in WANT if n in got)
print('TOOLS    :', TOOLS)

## 4. Weights (cached on Drive)

In [ ]:
!mkdir -p weights
W = 'weights/yolov8m-doclaynet.pt'
if not os.path.exists(WEIGHTS + '/yolov8m-doclaynet.pt'):
    !wget -q --show-progress -O "{WEIGHTS}/yolov8m-doclaynet.pt" \
       https://huggingface.co/hantian/yolo-doclaynet/resolve/main/yolov8m-doclaynet.pt
!cp -n "{WEIGHTS}/yolov8m-doclaynet.pt" {W}
!ls -la weights/

# Confirm YOLO lands on the GPU. Nothing in docstruct/ sets a device; ultralytics and
# sentence-transformers auto-select CUDA. If this prints cpu, stop and say so.
from docstruct.model.detector import ModelDetector
_m = ModelDetector(weights=W)._ensure_model()
print('YOLO device:', next(_m.model.parameters()).device)

## 5. Corpora

Both are fetched **into Drive** and then copied to local disk. Drive is the source of
truth so a dead session never re-downloads; local disk is what the runs read, because
the benchmark reads every page repeatedly and Drive's FUSE mount is slow for that.

In [ ]:
# FinanceBench - 84 SEC filings, 189 human-annotated evidence regions, CC-BY-NC-4.0.
!mkdir -p data/financebench "{CORPORA}/financebench"
!cp -n "{CORPORA}/financebench/"*.pdf data/financebench/ 2>/dev/null || true
!python scripts/fetch_financebench.py
!cp -n data/financebench/*.pdf "{CORPORA}/financebench/" 2>/dev/null || true

import glob, json
print(len(glob.glob('data/financebench/*.pdf')), 'PDFs (expect 84)')
print(len(json.load(open('data/qa/financebench.json'))), 'gold rows (expect 189)')

In [ ]:
# PMC papers - 133 open-access papers across 7 journals, each with the publisher's JATS.
!mkdir -p data/pmc "{CORPORA}/pmc"
!cp -n "{CORPORA}/pmc/"* data/pmc/ 2>/dev/null || true
!python scripts/fetch_pmc.py --per-journal 20
!cp -n data/pmc/* "{CORPORA}/pmc/" 2>/dev/null || true
!python scripts/build_jats_gold.py

## 6. Smoke — three papers before the full corpus

Smoke what is about to run. Five failures in an earlier session were invisible to a
green test suite and only surfaced by running the real CLI.

This used to chunk two 10-Ks to validate the FinanceBench run. That run is off by
default now (section 7), so the smoke would have spent ~7 minutes proving something
unused and printing a `MRR=0.0` that looks like a failure but is the measured result.

In [ ]:
# Three documents through the exact path section 8 uses.
!python scripts/section_reachability.py --limit 3 --out /content/smoke_reach.json
!python scripts/score_sections.py --limit 3 --tools docstruct_geo,langchain     --ckpt-dir /content/smoke_ckpt --out /content/smoke_sections.json     --report-md /content/smoke_sections.md

In [ ]:
import json
r = json.load(open('/content/smoke_reach.json'))
s = json.load(open('/content/smoke_sections.json'))
print('reachability ceiling (body):', r.get('body_ceiling_pct'), '%')
for name, v in s['results'].items():
    print(f"  {name:16} WindowDiff={v['windowdiff']}  Pk={v['pk']}  docs={v['n_docs']}  errors={v['errors']}")

assert s['results'], 'no tool scored a single document'
assert all(v['n_docs'] > 0 for v in s['results'].values()), 'a tool scored zero documents'
assert r.get('body_ceiling_pct', 0) > 50, 'gold is mostly unreachable in the PDF text'
print()
print('smoke ok')

## 7. FinanceBench — **off by default, and here is the measurement why**

Set `RUN_FINANCEBENCH = True` below only if you want to reproduce the null result.

Measured 2026-08-13 on two filings, 5 questions, chunking done once so the embedder
was the only variable (`notes.md` Stage 19). Rank of the first *relevant* chunk under
hybrid dense + BM25:

| embedder | ranks | recall@5 | recall@100 | MRR@5 |
|---|---|---|---|---|
| all-MiniLM-L6-v2 | 81, 83, 163, 239, 299 | 0/5 | 2/5 | **0.0000** |
| BAAI/bge-small-en-v1.5 | 63, 146, 173, 173, 181 | 0/5 | 1/5 | **0.0000** |
| intfloat/e5-small-v2 | 42, 62, 128, 128, 175 | 0/5 | 2/5 | **0.0000** |

The scoring rule is not at fault — a relevant chunk exists at region overlap 0.99–1.00.
Retrieval simply never reaches it. FinanceBench questions are analyst prompts with
almost no lexical overlap with the evidence, the evidence is a numeric table that small
encoders embed poorly, and a 10-K yields 729–1,176 chunks.

So every tool would score ~0. That discriminates nothing between chunkers, which is the
only thing this benchmark exists to measure, and a table of zeros reads as a chunking
result rather than a retrieval limit. **Section 8 is the run that matters.**

In [ ]:
# Off by default: ~8 GPU-hours to reproduce a table of zeros. See the markdown above.
RUN_FINANCEBENCH = False

if RUN_FINANCEBENCH:
    !mkdir -p reports
    !python -m docstruct.cli benchmark --pdfs-dir data/financebench --qa data/qa/financebench.json --weights {W} --tools {TOOLS} --relevance region --dump-scores --cache-dir "{CACHE}" --report-md reports/fb_report_region.md --report-json reports/fb_results_region.json

    # The dumped scores are the only reason the threshold sweep in section 9 can run
    # without a second GPU session, so check them here rather than discovering it
    # afterwards. This belongs inside the branch: with the flag off there is no file.
    import json as _json
    _d = _json.load(open('reports/fb_results_region.json'))
    assert any('hyb_scores' in q for t in _d['results'] for q in t['per_question']), (
        'benchmark ran without --dump-scores; section 9 would have nothing to read')
    print('dumped scores present')
else:
    print('FinanceBench retrieval skipped (RUN_FINANCEBENCH = False).')
    print('Its evidence is unreachable at top-5 under three embedders, so every tool')
    print('scores 0.0 -- that measures the retriever, not the chunkers.')
    print('It stays in the paper for parse fidelity and borderless-table detection.')
    print('notes.md Stage 19 has the measurement.')

In [ ]:
# Only present if the gated run above actually executed.
from IPython.display import Markdown, display
import os

if os.path.exists('reports/fb_report_region.md'):
    !cp -f reports/fb_report_region.md reports/fb_results_region.json "{REPORTS}/"
    display(Markdown(open('reports/fb_report_region.md').read()[:4000]))
else:
    print('no FinanceBench report - the run is off by default. Section 8 is the real work.')

## 8. Section-boundary agreement on the PMC papers

Does a chunker split where the document splits? Pk and WindowDiff against the
publisher's own JATS section boundaries — **lower is better**, 0.0 is perfect agreement.
Unlike section paths this is a real comparison: every chunker has boundaries.

The ceiling runs first. A gold boundary that cannot be found in the PDF's own text
cannot be scored against, and a Pk that quietly skipped a third of them would still look
like a result.

In [ ]:
!python scripts/section_reachability.py
!cp -f reports/section_reachability.json "{REPORTS}/" 2>/dev/null || true

In [ ]:
# Includes hybrid docstruct, which is why this wants a GPU: one figure-dense paper
# measured 475s geometry-only on a laptop CPU (notes.md Stage 18).
!python scripts/score_sections.py \
   --tools {TOOLS} --weights {W} --cache-dir "{CACHE}"
!cp -f reports/section_scores.md reports/section_scores.json "{REPORTS}/" 2>/dev/null || true
from IPython.display import Markdown, display
if os.path.exists('reports/section_scores.md'):
    display(Markdown(open('reports/section_scores.md').read()))

## 9. Sweep the region threshold — offline, seconds, no GPU

`RELEVANCE_REGION_MIN_OVERLAP = 0.7` is `# unvalidated`, and both the FinanceBench
leaderboard above and DocStruct's OHR-Bench region win rest on it. Because
`--dump-scores` recorded the continuous score behind every retrieved chunk, the
threshold is now a re-scoring rather than a re-run.

Read it for the **plateau**, not the peak: at 0.0 every chunk counts as relevant and MRR
is 1.0 by definition, so a metric that climbs as the threshold falls is not evidence for
a low threshold. The line that matters is whether the *ranking* moves.

In [ ]:
# Only meaningful if the FinanceBench run above actually produced results.
import os
if os.path.exists('reports/fb_results_region.json'):
    !python scripts/sweep_relevance_threshold.py        --results reports/fb_results_region.json        --out reports/fb_threshold_sweep.json
    !cp -f reports/fb_threshold_sweep.json "{REPORTS}/" 2>/dev/null || true
else:
    print('no FinanceBench results to sweep - RUN_FINANCEBENCH was False.')
    print('RELEVANCE_REGION_MIN_OVERLAP stays unvalidated; it needs a corpus whose')
    print('evidence is actually retrievable, so OHR-Bench region is the place to do it.')

## 10. Collect everything

In [ ]:
!cd reports && zip -q -r /content/docstruct_results.zip \
    fb_report_region.md fb_results_region.json fb_threshold_sweep.json \
    section_scores.md section_scores.json section_reachability.json 2>/dev/null || true
!cp -f /content/docstruct_results.zip "{BENCH}/" 2>/dev/null || true
!ls -la "{REPORTS}/"
try:
    from google.colab import files
    files.download('/content/docstruct_results.zip')
except Exception as e:
    print('browser download skipped:', e)
    print('the zip is on Drive at', BENCH)

---
## Back on the laptop

Commit the JSONs — they are the paper's evidence — and then:

1. **Read `section_reachability.json` before `section_scores.md`.** The ceiling is 84.5%
   of body sections; scores mean nothing against an assumed 100%.
2. **Treat the section table as provisional.** That metric found six defects in itself
   before producing a usable number — LaTeX-polluted gold, a collapsing aligner, a
   word-token spine, forward-only poisoning, punctuation sensitivity, and monotonicity
   applied to chunks. Five of the six made a competitor look worse than it is. The
   retrieval numbers have months of scrutiny; these have one session.
3. **`RELEVANCE_REGION_MIN_OVERLAP` is still unvalidated.** The offline sweep exists and
   works, but it needs a corpus whose evidence is actually retrievable — so the place to
   run it is OHR-Bench under `--relevance region`, not FinanceBench.